In [1]:
"""
================================================================================
STEP 1: EXTRACT BERT EMBEDDINGS FOR GCN
================================================================================
Extracts CLS embeddings from trained BERT model for ALL documents.
Output: node_features.npy (N × 768) in original dataset order
================================================================================
"""

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("EXTRACTING BERT EMBEDDINGS FOR GCN")
print("="*80)


EXTRACTING BERT EMBEDDINGS FOR GCN


In [2]:
# ============================================================================
# CONFIGURATION (must match training)
# ============================================================================
class Config:
    BERT_MODEL = 'google/muril-base-cased'
    BERT_DIM = 768
    HIDDEN_DIM = 256
    DROPOUT = 0.3
    MAX_LENGTH = 256
    BATCH_SIZE = 32  # Larger for inference
    FREEZE_BERT_LAYERS = 8
    
    # Paths
    DATA_PATH = 'preprocessed_news02.csv'
    MODEL_PATH = 'best_bert_only_model.pt'
    OUTPUT_PATH = 'node_features.npy'
    
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Device: {Config.DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [3]:
# ============================================================================
# MODEL (same architecture as training)
# ============================================================================
class BERTClassifier(nn.Module):
    def __init__(self, bert_model, num_labels, hidden_dim=256, dropout=0.3):
        super().__init__()
        self.bert = bert_model
        
        if Config.FREEZE_BERT_LAYERS > 0:
            for param in self.bert.embeddings.parameters():
                param.requires_grad = False
            for layer in self.bert.encoder.layer[:Config.FREEZE_BERT_LAYERS]:
                for param in layer.parameters():
                    param.requires_grad = False
        
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(Config.BERT_DIM, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_labels)
        )
    
    def forward(self, input_ids, attention_mask, return_embeddings=False):
        """
        If return_embeddings=True, returns CLS embeddings instead of logits
        """
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )
        
        # Get CLS token embedding (same as used in training)
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        
        if return_embeddings:
            return cls_embedding
        else:
            cls_output = self.dropout(cls_embedding)
            logits = self.classifier(cls_output)
            return logits

In [4]:
# ============================================================================
# DATASET
# ============================================================================
class NewsDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'idx': idx  # Keep track of original index
        }

In [5]:

# ============================================================================
# MAIN EXTRACTION
# ============================================================================
def extractor():
    print("\n" + "="*80)
    print("STEP 1: LOAD DATA")
    print("="*80)
    
    # Load data
    df = pd.read_csv(Config.DATA_PATH)
    print(f"✓ Loaded: {len(df):,} documents")
    
    # Load labels for metadata
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    if 'clean_categories' in df.columns:
        labels = le.fit_transform(df['clean_categories'].astype(str))
    else:
        labels = le.fit_transform(df['category'].astype(str))
    
    num_labels = len(le.classes_)
    print(f"✓ Categories: {num_labels}")
    print(f"✓ Classes: {list(le.classes_)}")
    
    # ========================================================================
    print("\n" + "="*80)
    print("STEP 2: LOAD TRAINED MODEL")
    print("="*80)
    
    # Load tokenizer and BERT
    tokenizer = AutoTokenizer.from_pretrained(Config.BERT_MODEL)
    bert_model = AutoModel.from_pretrained(Config.BERT_MODEL)
    
    # Create model
    model = BERTClassifier(
        bert_model=bert_model,
        num_labels=num_labels,
        hidden_dim=Config.HIDDEN_DIM,
        dropout=Config.DROPOUT
    ).to(Config.DEVICE)
    
    # Load trained weights
    print(f"Loading weights from: {Config.MODEL_PATH}")
    checkpoint = torch.load(Config.MODEL_PATH, map_location=Config.DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()  # Set to evaluation mode
    
    print(f"✓ Loaded model from epoch {checkpoint['epoch']+1}")
    print(f"✓ Model val F1: {checkpoint['val_f1']:.4f}")
    
    # ========================================================================
    print("\n" + "="*80)
    print("STEP 3: EXTRACT EMBEDDINGS")
    print("="*80)
    
    # Create dataset (ALL documents in original order)
    dataset = NewsDataset(df['content'].values, tokenizer, Config.MAX_LENGTH)
    dataloader = DataLoader(
        dataset,
        batch_size=Config.BATCH_SIZE,
        shuffle=False,  # CRITICAL: Keep original order!
        num_workers=0
    )
    
    print(f"✓ Processing {len(dataset):,} documents")
    print(f"✓ Batch size: {Config.BATCH_SIZE}")
    print(f"✓ Total batches: {len(dataloader)}")
    
    # Extract embeddings
    all_embeddings = []
    all_indices = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Extracting embeddings'):
            input_ids = batch['input_ids'].to(Config.DEVICE)
            attention_mask = batch['attention_mask'].to(Config.DEVICE)
            indices = batch['idx'].cpu().numpy()
            
            # Get CLS embeddings
            embeddings = model(input_ids, attention_mask, return_embeddings=True)
            
            all_embeddings.append(embeddings.cpu().numpy())
            all_indices.append(indices)
    
    # Combine all batches
    embeddings = np.vstack(all_embeddings)
    indices = np.concatenate(all_indices)
    
    print(f"\n✓ Extracted embeddings shape: {embeddings.shape}")
    print(f"✓ Expected shape: ({len(df)}, {Config.BERT_DIM})")
    
    # ========================================================================
    print("\n" + "="*80)
    print("STEP 4: VERIFY & SAVE")
    print("="*80)
    
    # Verify correctness
    assert embeddings.shape[0] == len(df), f"Shape mismatch! Got {embeddings.shape[0]}, expected {len(df)}"
    assert embeddings.shape[1] == Config.BERT_DIM, f"Dim mismatch! Got {embeddings.shape[1]}, expected {Config.BERT_DIM}"
    assert np.array_equal(indices, np.arange(len(df))), "Index order mismatch!"
    print("✓ All verification checks passed!")
    
    # Check for NaN/Inf
    if np.isnan(embeddings).any():
        print("⚠️  WARNING: NaN values detected!")
    elif np.isinf(embeddings).any():
        print("⚠️  WARNING: Inf values detected!")
    else:
        print("✓ No NaN/Inf values")
    
    # Statistics
    print(f"\nEmbedding statistics:")
    print(f"  Mean: {embeddings.mean():.4f}")
    print(f"  Std:  {embeddings.std():.4f}")
    print(f"  Min:  {embeddings.min():.4f}")
    print(f"  Max:  {embeddings.max():.4f}")
    
    # Save embeddings
    np.save(Config.OUTPUT_PATH, embeddings.astype(np.float32))
    print(f"\n✅ Saved embeddings to: {Config.OUTPUT_PATH}")
    
    # Also save labels for convenience
    np.save('labels.npy', labels.astype(np.int64))
    print(f"✅ Saved labels to: labels.npy")
    
    # Save metadata
    import json
    metadata = {
        'num_documents': int(len(df)),
        'embedding_dim': int(Config.BERT_DIM),
        'num_classes': int(num_labels),
        'class_names': le.classes_.tolist(),
        'model_epoch': int(checkpoint['epoch'] + 1),
        'model_val_f1': float(checkpoint['val_f1']),
        'bert_model': Config.BERT_MODEL,
        'max_length': Config.MAX_LENGTH
    }
    
    with open('embedding_metadata.json', 'w') as f:
        json.dump(metadata, f, indent=2)
    print(f"✅ Saved metadata to: embedding_metadata.json")
    
    # ========================================================================
    print("\n" + "="*80)
    print("EXTRACTION COMPLETE!")
    print("="*80)
    print("\n📁 Files created:")
    print(f"  1. {Config.OUTPUT_PATH} - Shape {embeddings.shape}")
    print(f"  2. labels.npy - Shape {labels.shape}")
    print(f"  3. embedding_metadata.json")
    print("\n✅ Ready for graph building (Step 2)!")
    print("="*80)

In [6]:
extractor()


STEP 1: LOAD DATA
✓ Loaded: 122,817 documents
✓ Categories: 10
✓ Classes: ['art', 'crime', 'economy', 'education', 'entertainment', 'global', 'health', 'politics', 'science and technology', 'sports']

STEP 2: LOAD TRAINED MODEL
Loading weights from: best_bert_only_model.pt
✓ Loaded model from epoch 12
✓ Model val F1: 0.8794

STEP 3: EXTRACT EMBEDDINGS
✓ Processing 122,817 documents
✓ Batch size: 32
✓ Total batches: 3839


Extracting embeddings: 100%|███████████████████████████████████████████████████████| 3839/3839 [29:10<00:00,  2.19it/s]



✓ Extracted embeddings shape: (122817, 768)
✓ Expected shape: (122817, 768)

STEP 4: VERIFY & SAVE
✓ All verification checks passed!
✓ No NaN/Inf values

Embedding statistics:
  Mean: -0.0020
  Std:  0.0220
  Min:  -0.4335
  Max:  0.1720

✅ Saved embeddings to: node_features.npy
✅ Saved labels to: labels.npy
✅ Saved metadata to: embedding_metadata.json

EXTRACTION COMPLETE!

📁 Files created:
  1. node_features.npy - Shape (122817, 768)
  2. labels.npy - Shape (122817,)
  3. embedding_metadata.json

✅ Ready for graph building (Step 2)!
